# XYZ Visualization in Jupyter

This notebook loads `.xyz` files and visualizes them with:
- ASE inline viewer (`x3d`)
- Optional ASE desktop GUI (`ase gui`)
- `py3Dmol` inline in notebook


In [ ]:
# If packages are missing, run this cell once:
# %pip install ase py3Dmol ipywidgets


In [1]:
from pathlib import Path
from io import StringIO
import subprocess

from ase.io import read, write
from ase.visualize import view
import py3Dmol

def find_xyz_files(root='.'):
    return sorted(Path(root).rglob('*.xyz'))

def load_xyz(path, index=':'):
    atoms = read(str(path), index=index)
    if isinstance(atoms, list):
        return atoms
    return [atoms]

def atoms_to_xyz_string(atoms):
    buf = StringIO()
    write(buf, atoms, format='xyz')
    return buf.getvalue()

def show_py3dmol(frames, frame=0, width=700, height=500, style='stick'):
    frame = max(0, min(frame, len(frames) - 1))
    xyz = atoms_to_xyz_string(frames[frame])
    v = py3Dmol.view(width=width, height=height)
    v.addModel(xyz, 'xyz')
    v.setStyle({style: {}})
    v.zoomTo()
    return v.show()

def animate_py3dmol(frames, width=700, height=500, style='stick'):
    combined = ''.join(atoms_to_xyz_string(a) for a in frames)
    v = py3Dmol.view(width=width, height=height)
    v.addModelsAsFrames(combined, 'xyz')
    v.setStyle({style: {}})
    v.zoomTo()
    v.animate({'loop': 'forward', 'reps': 0, 'step': 1, 'interval': 100})
    return v.show()

def open_ase_gui(path):
    # Opens the desktop ASE GUI (not inline).
    return subprocess.Popen(['ase', 'gui', str(path)])


In [2]:
# 1) Discover xyz files under current folder
# xyz_files = find_xyz_files('.')
# print(f'Found {len(xyz_files)} xyz file(s).')
# for i, p in enumerate(xyz_files[:20]):
#     print(f'[{i}] {p}')

# 2) Pick file (set this manually if needed)
# xyz_path = xyz_files[0]
xyz_path = Path('../outputs_xyz/ts_00000.xyz')
print('Selected:', xyz_path)


Selected: ../outputs_xyz/ts_00000.xyz


In [3]:
frames = load_xyz(xyz_path, index=':')
print(f'Loaded {len(frames)} frame(s) from {xyz_path}')


Loaded 1 frame(s) from ../outputs_xyz/ts_00000.xyz


In [4]:
# ASE inline viewer in notebook
# Works in Jupyter; if your environment does not render x3d, use py3Dmol below.
view(frames[0], viewer='x3d')


In [ ]:
# Optional: open desktop ASE GUI
# open_ase_gui(xyz_path)


In [5]:
# py3Dmol inline (single frame)
show_py3dmol(frames, frame=0, style='stick')


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# py3Dmol animation across all frames (for trajectories)
# animate_py3dmol(frames, style='stick')
